# lang-geom on Colab
Secrets needed (Colab ▸ 🔑): `HF_TOKEN` (with FLORES+ and Llama-3.1 gates accepted), `GITHUB_TOKEN` (repo scope). Runtime: A100 or L4.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
GH = userdata.get('GITHUB_TOKEN')
REPO = 'Dadhichi/Language-Geometry'
BRANCH = 'claude/compassionate-newton-vufc0c'   # working branch shared with Claude Code
DATA = '/content/drive/MyDrive/lang_geom'
os.makedirs(DATA, exist_ok=True)

In [ ]:
%cd /content
import os
if not os.path.exists('Language-Geometry'):
    !git clone -b {BRANCH} https://{GH}@github.com/{REPO}.git Language-Geometry
%cd /content/Language-Geometry
!git pull origin {BRANCH}
!pip -q install -r requirements.txt
!python fit.py --selftest | tail -16

## Smoke test (3 languages, 16 sentences) — fix anything that breaks here before the full run
fit.py needs ≥ 3 languages (composition and pivot tests use triples).

In [ ]:
!python extract.py --model Qwen/Qwen2.5-7B --tag smoke --out /content/scratch_out --langs eng_Latn,deu_Latn,zho_Hans --smoke 16 --overwrite
!python fit.py --data /content/scratch_out/smoke --fit_split dev --test_splits devtest --k 8 --pca per_lang --device cuda --null_reps 1 --layers 0,4 --out /content/scratch_out/smoke_results
!head -3 /content/scratch_out/smoke_results/summary.csv

## Full FLORES extraction

In [ ]:
!python extract.py --model Qwen/Qwen2.5-7B --tag qwen25_7b --out $DATA --flores hf

## Fit (per-language PCA; FLORES dev fit, devtest test)

In [ ]:
for pooling in ['mean', 'last']:
    !python fit.py --data $DATA/qwen25_7b --pooling {pooling} --fit_split dev --test_splits devtest \
        --k 64,128,256 --pca per_lang --device cuda --null_reps 2 --out results/qwen25_7b_{pooling}

## Optional: TED splits (n_fit ≫ k), then rerun with larger k / full d

In [ ]:
!python build_ted.py --out /content/flores200_dataset --n_fit 6000 --n_test 1000
!python extract.py --model Qwen/Qwen2.5-7B --tag qwen25_7b --out $DATA --flores hf --local_dir /content/flores200_dataset --splits tedfit,tedtest
!python fit.py --data $DATA/qwen25_7b --pooling mean --fit_split tedfit --test_splits devtest,tedtest \
    --k 256,512,full --pca per_lang --device cuda --null_reps 2 --out results/qwen25_7b_mean_ted

## Push results (CSVs only; activations stay on Drive)

In [ ]:
!git config user.email 'colab@example.com' && git config user.name 'colab'
!git add results && git commit -m "results: $(date +%F)" || true
!git pull --rebase origin {BRANCH}
!git push https://{GH}@github.com/{REPO}.git HEAD:{BRANCH}